In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr 
from scipy import stats

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_slow")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_slow")

#EEG_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")
#PUPIL_ROOT = Path(r"C:/Users/cdd/Documents/Uni/Special_course/pupil_processed_clara")   



FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

OUT_TRIALS  = Path("trial_level_cca_slow.csv")
OUT_SUBJECT = Path("subject_level_cca_slow.csv")
OUT_WEIGHTS = Path("cca_weights_slow")

#SUBJECTS = np.setdiff1d(np.arange(35, 59), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
SHIFTS = np.arange(-100, 101)      # ±1 s at 100 Hz → samples
WIN_OFFSET1 = 200                  # discard first 2 s
WIN_OFFSET2 = 101                  # discard last 1.01 s
win = slice(WIN_OFFSET1, -WIN_OFFSET2)

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(eeg, pupil)
    u, v = cca.transform(eeg, pupil)
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])

def candidate_lags_units(step_ms=100, max_ms=2000):
    # Your code uses 10 ms units (lag_ms = shift*10)
    step_units = step_ms // 10
    max_units  = max_ms  // 10
    return list(range(-max_units, max_units + 1, step_units))


In [2]:
def load_all_trials(
        sub: int,
        eeg_root: Path = EEG_ROOT,
        pupil_root: Path = PUPIL_ROOT,
        min_len: int = 40,
        max_len_diff: int = 30,
) -> list[tuple[np.ndarray, np.ndarray, dict]]:
    """
    Load *all* valid EEG-pupil trial pairs for one subject.

    Parameters
    ----------
    sub : int
        Numeric subject ID (e.g. 42).
    eeg_root, pupil_root : Path
        Roots of the pre-processed EEG and pupil folders.
    min_len : int
        Minimum number of samples a pupil trace must have to be accepted.
    max_len_diff : int
        Reject trial if |len(pupil)-len(eeg)| exceeds this.
    Returns
    -------
    trials : list of (eeg, pupil_z, meta)
        * eeg        - (T × n_channels) float64, already centred/scaled
        * pupil_z    - (T × 1) float64, per-trial z-scored
        * meta       - dict with subject/condition/load/epoch
    """
    trials = []
    sub_tag = f"sub-{sub:03d}"
    eeg_sub  = eeg_root   / sub_tag
    pupil_sub = pupil_root / sub_tag

    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        return trials

    # iterate condition (“control” / “memory”) and load (“05” / “09” / “13”)
    for cond_path in sorted(eeg_sub.iterdir()):
        if not cond_path.is_dir():
            continue
        for load_path in sorted(cond_path.iterdir()):
            if not load_path.is_dir():
                continue

            # matching pupil directory
            pupil_path = pupil_sub / cond_path.name / load_path.name
            if not pupil_path.exists():
                continue

            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))
            common = {f.name for f in eeg_epochs} & {f.name for f in pupil_epochs}
            if not common:
                continue

            for fname in sorted(common):
                eeg_df = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#", skiprows=1,
                                       names=["time", "diameter_z"], index_col=0)

                eeg   = eeg_df.values.astype(float)
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic validity checks
                if len(pupil) < min_len or abs(len(pupil) - len(eeg)) > max_len_diff:
                    continue

                # normalise signals ---------------------------------------- now do it after trimming shifted trials
                #eeg_norm = normalise_eeg(eeg)           # your helper from before
                #pupil_z  = ((pupil - pupil.mean()) / pupil.std(ddof=0))

                # same number of samples
                T = min(len(eeg), len(pupil))
                eeg = eeg[0:T, :]  # (T × n_channels)
                pupil  = pupil[0:T].reshape(-1, 1)

                meta = {
                    "subject":   sub_tag,
                    "condition": cond_path.name,
                    "load":      int(load_path.name),
                    "epoch":     fname
                }
                trials.append((eeg, pupil, meta))

    return trials

from typing import List, Tuple
import numpy as np

def split_trials_by_condition(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        memory_label: str = "memory",
        control_label: str = "control"
) -> Tuple[List[Tuple[np.ndarray, np.ndarray, dict]], List[Tuple[np.ndarray, np.ndarray, dict]]]:
    """
    Separate a mixed list of (eeg, pupil, meta) trial tuples into memory-condition and control-condition sub-lists.

    Parameters
    ----------
    trials : list of tuples
        Each tuple = (eeg_array, pupil_array, meta_dict).
        meta_dict must contain a key 'condition'.
    memory_label : str
        The value of meta['condition'] that marks a memory trial.
    control_label : str
        The value of meta['condition'] that marks a control trial.

    Returns
    -------
    memory_trials  : list[tuple]
    control_trials : list[tuple]
    """
    memory_trials  = []
    control_trials = []

    for eeg, pupil, meta in trials:
        cond = meta.get("condition", "").lower()
        if cond == memory_label:
            memory_trials.append((eeg, pupil, meta))
        elif cond == control_label:
            control_trials.append((eeg, pupil, meta))
        else: raise ValueError(f"Unknown condition label: {cond}")

    return memory_trials, control_trials


In [3]:

from typing import List, Tuple, Optional, Dict

# trials  : list of (eeg, pupil_z, meta)   -- the tuples returned by load_all_trials
# shift   : integer sample shift (best_shift)
# win     : slice or None                  -- cropping window (set to None if the
#                                            trials are already pre-trimmed)
def concat_trials(
    trials: List[Tuple[np.ndarray, np.ndarray, dict]],
    shift: int = 0,  # samples; positive = EEG delayed (EEG after pupil)
    trim=None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Concatenate trials for CCA without circular wrap.

    Returns
    -------
    X : ndarray, shape (Σ T_eff, n_channels)   # EEG
    Y : ndarray, shape (Σ T_eff, 1)            # pupil
    """
    X_blocks, Y_blocks = [], []
    if trim is not None:
        target_trim_samp = trim // 10   # 5000 ms → 500 samples at 100 Hz
        extra_samp = max(0, (target_trim_samp - abs(shift)) // 2)
        print(f"Lag is {shift} samples. Trimming {extra_samp} samples from each end after lag adjustment")

    for eeg, pupil_z, _ in trials:
        # Ensure 1D/2D shapes: EEG (T, C), pupil (T, 1)
        if eeg.ndim == 1:
            eeg = eeg[:, None]
        if pupil_z.ndim == 1:
            pupil_z = pupil_z[:, None]

        T = min(len(eeg), len(pupil_z))
        if T == 0:
            print("Warning: zero-length trial encountered - skipped")
            continue

        # Optional common window first (keeps boundaries away)
        start = WIN_OFFSET1
        stop  = T - WIN_OFFSET2
        if stop <= start:
            print("Warning: collapsed window after trimming - skipped")
            continue  # window collapsed; skip this trial

        eeg_w   = eeg[start:stop]
        pupil_w = pupil_z[start:stop]
        Tw = len(eeg_w)

        # Apply lag by asymmetric trimming (no roll, no wrap)
        # Positive shift => EEG delayed => drop 'shift' from EEG start and pupil end
        # Negative shift => EEG advanced => drop '-shift' from pupil start and EEG end
        if shift >= 0:
            if Tw <= shift: continue  # nothing left after lag
            eeg_seg   = eeg_w[shift:]             # drop early EEG
            pupil_seg = pupil_w[:Tw - shift]      # drop late pupil
        else:
            s = -shift
            if Tw <= s: continue
            eeg_seg   = eeg_w[:Tw - s]            # drop late EEG
            pupil_seg = pupil_w[s:]               # drop early pupil

        if trim is not None:
            Lcur = int(len(eeg_seg))
            if extra_samp > 0:
                extra_eff = int(min(extra_samp, max(0, (Lcur - 2) // 2)))
                if extra_eff > 0:
                    eeg_seg   = eeg_seg[extra_eff : Lcur - extra_eff]
                    pupil_seg = pupil_seg[extra_eff : Lcur - extra_eff]

        eeg_seg = normalise_eeg(eeg_seg)           # your helper from before
        pupil_seg = ((pupil_seg - pupil_seg.mean()) / pupil_seg.std(ddof=0))

        # Shapes now match
        X_blocks.append(eeg_seg)
        Y_blocks.append(pupil_seg)

    if not X_blocks:
        print("Warning: no valid trials after concatenation - returning empty arrays")
        return np.empty((0, trials[0][0].shape[-1])), np.empty((0, 1))

    X = np.vstack(X_blocks)
    Y = np.vstack(Y_blocks)
    return X, Y

def iterate_trials(trials, shift):
    """
    Yield (eeg_aligned, pupil_aligned, meta) one by one, with no circular wrap.

    Conventions
    -----------
    - `shift` in samples, along time axis (axis=0).
    - shift >= 0  → EEG delayed  → drop early EEG, drop late pupil
    - shift <  0  → EEG advanced → drop late EEG, drop early pupil
    """
    for eeg, pupil_z, meta in trials:
        # Window first
        T = min(len(eeg), len(pupil_z))
        start = WIN_OFFSET1
        stop  = T - 50
        if stop <= start:
            print("Warning: collapsed window after trimming - skipped")
            continue  # window collapsed; skip this trial

        eeg_w   = eeg[start:stop]
        pupil_w = pupil_z[start:stop]

        # Ensure both have time along axis 0 and share the same window length
        Tw = len(eeg_w)
        
        if Tw == 0: continue

        if shift >= 0: # EEG delayed: drop early EEG, drop late pupil
            eeg_out   = eeg_w[shift:]           # works for 1D or 2D (time on axis 0)
            pupil_out = pupil_w[:Tw - shift]
        else:
            s = -shift # EEG advanced: drop late EEG, drop early pupil
            
            eeg_out   = eeg_w[:Tw - s]
            pupil_out = pupil_w[s:]

        eeg_out = normalise_eeg(eeg_out)           # your helper from before
        pupil_out = ((pupil_out - pupil_out.mean()) / pupil_out.std(ddof=0))

        yield eeg_out, pupil_out, meta.copy()

import pandas as pd
import json
from pathlib import Path

def save_cca_weights(cca, subject_tag, lag_ms, eeg_ch_names, condition, out_dir=Path("weights")):
    """
    Dump EEG & pupil canonical weights to CSV/JSON for one subject.

    Parameters
    ----------
    cca            : fitted sklearn.cross_decomposition.CCA
    subject_tag    : "sub-042"
    lag_ms         : e.g. -90
    eeg_ch_names   : list[str] same order as columns in your trial matrices
    out_dir        : destination folder (created if missing)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # ------- Pupil weight -> JSON ---------------------------------------
    w_pupil_og = float(cca.y_weights_[0, 0])   # scalar in 1-dim pupil case
    w_pupil = 1.0
    with open(out_dir / f"{subject_tag}_pupil_weight_{condition}_{lag_ms:+d}ms.json", "w") as fh:
        json.dump({"weight": w_pupil}, fh, indent=2)
    
    # ------- EEG weights -> tidy CSV ------------------------------------
    w_eeg = cca.x_weights_[:, 0] / w_pupil_og
    w_eeg = pd.Series(w_eeg, index=eeg_ch_names, name="weight")
    w_eeg.index.name = "channel"
    w_eeg.to_csv(out_dir / f"{subject_tag}_eeg_weights_{condition}_{lag_ms:+d}ms.csv")

    print(f"saved weights for {subject_tag} (lag {lag_ms:+d} ms)")

def search_best_lag(
            train_trials: List[Tuple[np.ndarray, np.ndarray, dict]],
            shifts: np.ndarray = SHIFTS,
            return_curve: bool = False,
            max_ms=None
    ) -> Tuple[float, int, Dict[int, float] | None]:
        
        r_per_shift: Dict[int, float] = {}
        for s in shifts:
            eeg, pupil = concat_trials(train_trials, shift=s, trim=max_ms)
            r_per_shift[s] = cca_corr(eeg, pupil)

        best_shift = max(r_per_shift, key=r_per_shift.get)
        best_corr  = r_per_shift[best_shift]

        if return_curve:
            return best_corr, best_shift, r_per_shift
        else:
            return best_corr, best_shift, None



In [4]:
%matplotlib qt  
# %matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
# Use Arial everywhere (falls back if not available)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,          # base font size
    "axes.titlesize": 16,     # figure/axes titles
    "axes.labelsize": 13,     # x/y labels
    "legend.fontsize": 11,    # legend text
    "xtick.labelsize": 11,    # tick labels
    "ytick.labelsize": 11,
    "figure.titlesize": 18,   # suptitle
})

subj = 75
trials = load_all_trials(subj)                         # list of (eeg, pupil)
trials_memory, trials_control = split_trials_by_condition(trials)

# ----- lag search on memory ------------------------------------------
return_curve = True  # set to True if you want the full curve
best_corr, best_shift, mean_r_per_shift = search_best_lag(trials_memory, candidate_lags_units(), return_curve)
print(f"Subject {subj:02d}: best lag {best_shift*10} ms with r = {best_corr:.3f}")
#best_shift_by_sub[subj] = best_shift      # samples, not ms

if return_curve:
    # -------------------------------------------------------------
    # 2) Build x and y vectors for plotting
    # -------------------------------------------------------------
    lags   = np.array(sorted(mean_r_per_shift))                 # x-axis (samples)
    r_mean = np.array([mean_r_per_shift[s] for s in lags])      # y-axis (mean r)

    # If you prefer milliseconds instead of samples:
    # fs = 1000  # replace with your real sampling rate
    # lags = lags * 1000 / fs

    # -------------------------------------------------------------
    # 3) Plot
    # -------------------------------------------------------------
    import matplotlib.pyplot as plt

    plt.figure(figsize=(6, 3.5))
    plt.plot(lags, r_mean, lw=2)
    plt.axvline(best_shift, ls='--', lw=1.5,
                label=f'best lag = {best_shift}')
    plt.xlabel('Lag (samples)')
    plt.ylabel('Mean CCA correlation (r)')
    plt.title('Mean CCA correlation vs. lag')
    plt.grid(alpha=.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

Subject 75: best lag -800 ms with r = 0.242


In [5]:
if return_curve:
    # -------------------------------------------------------------
    # 2) Build x and y vectors for plotting
    # -------------------------------------------------------------
    lags   = np.array(sorted(mean_r_per_shift))                 # x-axis (samples)
    r_mean = np.array([mean_r_per_shift[s] for s in lags])      # y-axis (mean r)

    # If you prefer milliseconds instead of samples:
    # fs = 1000  # replace with your real sampling rate
    # lags = lags * 1000 / fs

    # -------------------------------------------------------------
    # 3) Plot
    # -------------------------------------------------------------
    import matplotlib.pyplot as plt

    lags_plot = -lags
    best_plot = -best_shift

    plt.figure(figsize=(4, 3.5))
    plt.plot(lags_plot, r_mean, lw=2)
    plt.axvline(best_plot, ls='--', lw=1.5, label=f'best lag = {best_plot}')
    plt.xlabel('Lag (samples)')

    plt.ylabel('r')
    plt.grid(alpha=.3)
    plt.legend()
    plt.tight_layout()

    #ax= plt.gca()
    #ax.invert_xaxis()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
DIG_LEN   = 200           # 2 s at 100 Hz
rows_trials = []
rows_subject = [] 
best_shift_by_sub = {} 

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)
    
    # ----- lag search on memory ------------------------------------------
    return_curve = True  # set to True if you want the full curve
    max_ms = 5000
    best_corr, best_shift, mean_r_per_shift = search_best_lag(trials_memory, candidate_lags_units(max_ms=max_ms), return_curve)
    print(f"Subject {subj:02d}: best lag {best_shift*10} ms with r = {best_corr:.3f}")
    best_shift_by_sub[subj] = best_shift      # samples, not ms

    if return_curve:
        # -------------------------------------------------------------
        # 2) Build x and y vectors for plotting
        # -------------------------------------------------------------
        lags   = np.array(sorted(mean_r_per_shift))                 # x-axis (samples)
        r_mean = np.array([mean_r_per_shift[s] for s in lags])      # y-axis (mean r)

        # If you prefer milliseconds instead of samples:
        # fs = 1000  # replace with your real sampling rate
        # lags = lags * 1000 / fs

        # -------------------------------------------------------------
        # 3) Plot
        # -------------------------------------------------------------
        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 3.5))
        plt.plot(lags, r_mean, lw=2)
        plt.axvline(best_shift, ls='--', lw=1.5,
                    label=f'best lag = {best_shift}')
        plt.xlabel('Lag (samples)')
        plt.ylabel('Mean canonical correlation (r)')
        plt.title('Mean CCA correlation vs. lag')
        plt.grid(alpha=.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
DIG_LEN   = 200           # 2 s at 100 Hz
rows_trials = []
rows_subject = [] 
best_shift_by_sub = {} 

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)
    
    # ----- lag search on memory ------------------------------------------
    return_curve = True  # set to True if you want the full curve
    max_ms = 7000
    best_corr, best_shift, mean_r_per_shift = search_best_lag(trials_memory, candidate_lags_units(max_ms=max_ms), return_curve, max_ms=max_ms)
    print(f"Subject {subj:02d}: best lag {best_shift*10} ms with r = {best_corr:.3f}")
    best_shift_by_sub[subj] = best_shift      # samples, not ms

    if return_curve:
        # -------------------------------------------------------------
        # 2) Build x and y vectors for plotting
        # -------------------------------------------------------------
        lags   = np.array(sorted(mean_r_per_shift))                 # x-axis (samples)
        r_mean = np.array([mean_r_per_shift[s] for s in lags])      # y-axis (mean r)

        # If you prefer milliseconds instead of samples:
        # fs = 1000  # replace with your real sampling rate
        # lags = lags * 1000 / fs

        # -------------------------------------------------------------
        # 3) Plot
        # -------------------------------------------------------------
        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 3.5))
        plt.plot(lags, r_mean, lw=2)
        plt.axvline(best_shift, ls='--', lw=1.5,
                    label=f'best lag = {best_shift}')
        plt.xlabel('Lag (samples)')
        plt.ylabel('Mean canonical correlation (r)')
        plt.title('Mean CCA correlation vs. lag')
        plt.grid(alpha=.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
DIG_LEN   = 200           # 2 s at 100 Hz
rows_trials = []
rows_subject = [] 
best_shift_by_sub = {} 

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)
    
    # ----- lag search on memory ------------------------------------------
    return_curve = False  # set to True if you want the full curve
    best_corr, best_shift, mean_r_per_shift = search_best_lag(trials_memory, candidate_lags_units(), return_curve)
    print(f"Subject {subj:02d}: best lag {best_shift*10} ms with r = {best_corr:.3f}")
    best_shift_by_sub[subj] = best_shift      # samples, not ms

    if return_curve:
        # -------------------------------------------------------------
        # 2) Build x and y vectors for plotting
        # -------------------------------------------------------------
        lags   = np.array(sorted(mean_r_per_shift))                 # x-axis (samples)
        r_mean = np.array([mean_r_per_shift[s] for s in lags])      # y-axis (mean r)

        # If you prefer milliseconds instead of samples:
        # fs = 1000  # replace with your real sampling rate
        # lags = lags * 1000 / fs

        # -------------------------------------------------------------
        # 3) Plot
        # -------------------------------------------------------------
        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 3.5))
        plt.plot(lags, r_mean, lw=2)
        plt.axvline(best_shift, ls='--', lw=1.5,
                    label=f'best lag = {best_shift}')
        plt.xlabel('Lag (samples)')
        plt.ylabel('Mean canonical correlation (r)')
        plt.title('Mean CCA correlation vs. lag')
        plt.grid(alpha=.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # ----- fit weights ONCE using all train trials at best_shift ---------
    best_shift = int(best_shift)  # convert to int if it was float
    X_mem, Y_mem = concat_trials(trials_memory, shift=best_shift)
    X_ctrl, Y_ctrl = concat_trials(trials_control, shift=best_shift)
    
    cca_mem = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca_mem.fit(X_mem, Y_mem)

    
    # -------------------------------------------------------------
    # 2) whole‑trial correlation for MEMORY
    # -------------------------------------------------------------
    cvX_m, cvY_m = cca_mem.transform(X_mem, Y_mem)
    r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "memory",
        "lag_ms":    best_shift * 10,
        "r":         float(r_mem),
        "p-value": float(p_mem),
    })

    # -------------------------------------------------------------
    # 3) whole‑trial correlation for CONTROL
    # -------------------------------------------------------------
    cvX_c, cvY_c = cca_mem.transform(X_ctrl, Y_ctrl)
    r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "control",
        "lag_ms":    best_shift * 10,
        "r":         float(r_ctrl),
        "p-value": float(p_ctrl), 
    })

    save_cca_weights(cca_mem, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE, out_dir=OUT_WEIGHTS)

    # ----- per-trial r on TEST with frozen lag & weights -----------------
    for eeg_s, pupil_s, meta in iterate_trials(trials_memory, best_shift):
        n_digits = meta['load']
        for d in range(n_digits):
            w = slice(d*DIG_LEN, (d+1)*DIG_LEN)

        #cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
        #cca.fit(eeg_s, pupil_s)
        cv_eeg, cv_pupil = cca_mem.transform(eeg_s, pupil_s)
        cv_eeg = cv_eeg[:, 0]  # first CCA component
        cv_pupil = cv_pupil[:, 0]  # first CCA component

        for d in range(n_digits):
            w = slice(d * DIG_LEN, (d + 1) * DIG_LEN)
            x, y = cv_eeg[w], cv_pupil[w]
            
            if x.size == 0 or y.size == 0:
                print("Empty slice:", meta['subject'], meta['epoch'], d)
                continue

            if not np.isfinite(x).all() or not np.isfinite(y).all():
                print("Non-finite values:", meta['subject'], meta['epoch'], d, "→", np.sum(~np.isfinite(x)), "bad in X;", np.sum(~np.isfinite(y)), "bad in Y")
                continue

            if np.std(x) == 0 or np.std(y) == 0:
                print("Const window:", meta['subject'], meta['epoch'], d, np.std(x), np.std(y))
                continue

            r = np.corrcoef(x, y)[0, 1]
            rows_trials.append({
                'subject': meta['subject'],
                'condition': meta['condition'],
                'load': meta['load'],
                'epoch': meta['epoch'],
                'lag_ms': best_shift * 10,  # convert samples to ms
                'digit_pos': d + 1,  # 1-based digit index
                'r': float(r),  # convert to plain float for JSON-ability
            })

        meta['r'] = float(np.corrcoef(cv_eeg, cv_pupil)[0,1])
        meta['lag_ms'] = best_shift *10
        all_rows.append(meta)

df_sameW = pd.DataFrame(rows_trials)
grand = (df_sameW.query("condition == 'memory'")
           .groupby(['load','digit_pos'])['r']
           .agg(['mean','sem'])
           .reset_index())

for L in [5,9,13]:
    sub = grand[grand.load == L]
    plt.errorbar(sub.digit_pos, sub['mean'],
                 yerr=sub['sem'], label=f"{L} digits")
plt.axhline(0, c='k', lw=.5)
plt.xlabel("Digit position in the sequence")
plt.ylabel("CCA correlation (mean ± SEM)")
plt.legend(); plt.tight_layout()

##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################
pd.DataFrame(all_rows).to_csv(OUT_TRIALS, index=False)
print(f"Finished - saved per-trial correlations with plain CCA as {OUT_TRIALS}.")

##############################################################################
# 4)  after the loop - save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv(OUT_SUBJECT, index=False)
print(f"Finished - saved subject-level correlations as {OUT_SUBJECT}.")


In [ ]:
# 1. Load the file
corr_sub = pd.read_csv(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\subject_level_cca_slow.csv")

# 2. Sort rows by the “r” column (descending) and keep the 10 largest
top10 = corr_sub.sort_values("r", ascending=False).head(10)

# 3. Do whatever you need with those lines
print(top10)   

# 1. Load the file
corr_sub = pd.read_csv(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\trial_level_cca_slow.csv")

# 2. Sort rows by the “r” column (descending) and keep the 10 largest
corr_sub9 = corr_sub[corr_sub["load"] == 9]
top10 = corr_sub9.sort_values("r", ascending=False).head(10)

# 3. Do whatever you need with those lines
print(top10)   

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.stats.anova as sm_anova

# keep only MEMORY trials and clip r to avoid ±∞ after Fisher-z
df_mem_sameW = (df_sameW.query("condition == 'memory'")
            .assign(z = lambda d: np.arctanh(
                d.r.clip(lower=-0.999_999, upper=0.999_999))))

loads   = sorted(df_mem_sameW.load.unique())      # [5, 9, 13]
digits  = sorted(df_mem_sameW.digit_pos.unique()) # 1 … 13

for L in loads:
    wide = (df_mem_sameW.query("load == @L")
                   .pivot_table(index="subject",
                                columns="digit_pos",
                                values="r",
                                aggfunc="mean")
                   .dropna())          # subject must have every digit

    long = wide.reset_index().melt(id_vars='subject',
                                   var_name='digit_pos',
                                   value_name='r')

    model = smf.ols("r ~ C(digit_pos) + C(subject)", data=long).fit()
    anova = sm_anova.anova_lm(model, typ=2)
    print(f"\n=== Load {L} digits ===")
    print(anova.loc["C(digit_pos)"])

    df_num = int(anova.loc["C(digit_pos)", "df"])   # k‑1
    df_den = int(anova.loc["Residual",     "df"])   # (k‑1)*(s‑1)

    Fval   = anova.loc["C(digit_pos)", "F"]
    pval   = anova.loc["C(digit_pos)", "PR(>F)"]

    print(f"F({df_num}, {df_den}) = {Fval:.2f},  p = {pval:.3g}")






In [ ]:
import pingouin as pg
import statsmodels.api as sm

for L in loads:
    long = (df_mem_sameW.query("load == @L")
            .pivot_table(index="subject", columns="digit_pos", values="z", aggfunc="mean")
            .dropna()
            .reset_index()
            .melt(id_vars="subject", var_name="digit_pos", value_name="z"))
    aov = pg.rm_anova(data=long, dv='z', within='digit_pos', subject='subject', detailed=True, correction=True)
    print(f"\n=== Load {L} RM-ANOVA (omnibus) ===")
    print(aov)  # the table line for digit_pos is your omnibus test



In [ ]:
import os, fnmatch, pandas as pd, numpy as np
%matplotlib qt  
# %matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
# Use Arial everywhere (falls back if not available)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,          # base font size
    "axes.titlesize": 16,     # figure/axes titles
    "axes.labelsize": 13,     # x/y labels
    "legend.fontsize": 11,    # legend text
    "xtick.labelsize": 11,    # tick labels
    "ytick.labelsize": 11,
    "figure.titlesize": 18,   # suptitle
})

# ---------------------------------------------------------------------
# helper: read the weight CSV that matches subject + condition + lag
# ---------------------------------------------------------------------
data = pd.read_csv(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\subject_level_cca_slow.csv")
best_shift_by_sub = data.groupby("subject")["lag_ms"].agg(lambda x: x.value_counts().index[0]).to_dict()

WEIGHT_DIR = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\cca_weights_slow"


def find_weight_file(subj: int,
                     base_dir: Path = Path(WEIGHT_DIR)
                     ) -> Path:
    """
    Find the CSV for given condition ('ctrl' or 'mem') for the chosen subject.
    If lag_ms is provided, match exactly. Otherwise, take any that matches.
    """
    subj_str = f"{subj:03d}"
    subj_glob = f"sub-{subj_str}_eeg_weights_mem_*.csv"
    candidates = list((base_dir).glob(subj_glob))
    if not candidates:
        return None
    if len(candidates) > 1:
        return None  # ambiguous
    return candidates[0]

def load_weights(subj: int):
    csv_path = find_weight_file(subj)
    if csv_path is None or not csv_path.exists():
        return None, None
    df = pd.read_csv(csv_path)
    ch_names = df['channel'].tolist()
    weights  = df['weight'].to_numpy()
    return ch_names, weights


def plot_trials(plot_sub, plot_trial, condition, lag_samples):
    fs = 100  # Hz
    lag_ms = int(lag_samples * 1000 / fs)

    # -- load the trial ---------------------------------------------------
    for eeg, pupil, meta in load_all_trials(plot_sub):
        if meta["epoch"] == plot_trial:
            break
    else:
        raise ValueError(f"{plot_trial} not found for subject {plot_sub}")

    # -- load weights & build weight vector -------------------------------
    ch_names, weights = load_weights(plot_sub)
    if ch_names is None or weights is None:
        print(f"Missing or ambiguous weights for subject {plot_sub:03d}, cannot plot.")
        return
    if ch_names != FRONTAL_MIDLINE:
        print("Warning: channel names in weights do not match expected frontal midline set.")

    # -- window ------------------------------------------------------------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    # -- project & window --------------------------------------------------
    eeg_w = np.dot(eeg, weights).ravel()[win]
    pupil_vec = pupil.ravel()[win]

    Tw = min(len(eeg_w), len(pupil_vec))
    if Tw == 0:
        raise ValueError("Empty window after applying WIN_OFFSETs.")
    if abs(lag_samples) >= Tw:
        raise ValueError(f"lag ({lag_samples}) is >= window length ({Tw}); nothing to align.")

    # -- asymmetric trim (no roll) ----------------------------------------
    if lag_samples >= 0:
        eeg_w     = eeg_w[lag_samples:]
        pupil_vec = pupil_vec[:Tw - lag_samples]
        t0 = 0.0
    else:
        s = -lag_samples
        eeg_w     = eeg_w[:Tw - s]
        pupil_vec = pupil_vec[s:]
        t0 = 0.0

    eeg_w = normalise_eeg(eeg_w)
    pupil_vec = (pupil_vec - pupil_vec.mean()) / pupil_vec.std(ddof=0)
    # -- time axis ---------------------------------------------------------
    T = len(eeg_w)
    t = t0 + (np.arange(T) / fs)

    # -- plot --------------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(4, 3))
    ax2 = ax1.twinx()

    ax1.plot(t, pupil_vec, alpha=.8, label="PPD")
    ax2.plot(t, eeg_w,     alpha=.7, label=r"$\theta \, a_s$", color="black")

    ax1.set_xlabel("Time [s]")
    ax1.set_ylabel("PPD (z)")
    ax2.set_ylabel(r"CCA-weighted EEG ($\theta \, a_s$)", color="black")

    ax1.set_xlim(t[0], t[-1])
    ax2.set_xlim(t[0], t[-1])

    ax1.set_yticks([])
    ax2.set_yticks([])


    # show onset line only if it's within x-lims
    if t[0] <= 0 <= t[-1]:
        ax1.axvline(0, ls="--", c="k", lw=.7)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    #fig.suptitle(
    #    f"Subject {plot_sub:03d} - {plot_trial}  |  {condition}, lag {lag_ms:+d} ms"
    #)
    fig.tight_layout()
    plt.show()

In [ ]:
plot_sub      = 58
plot_trial    = "trial_136.csv"
condition     = "memory"          # or "control"
lag_samples   = best_shift_by_sub.get(plot_sub, 0) // 10  

plot_trials(plot_sub, plot_trial, condition, lag_samples) # r = 0,96

In [ ]:
plot_sub      = 58
plot_trial    = "trial_078.csv"
condition     = "memory"         
lag_samples   = best_shift_by_sub.get(plot_sub, 0) // 10 

plot_trials(plot_sub, plot_trial, condition, lag_samples) # r = -0,95

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PLOTS_DIR = Path(r"C:\Users\cdd\OneDrive - Danmarks Tekniske Universitet\Spring2025\Special_course\plots_appC")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)      # creates the folder tree if it doesn't exist

def trial_average2(sub_id, trials, load, cond, lag_samples=0, channels=FRONTAL_MIDLINE):
    """
    Return average pupil (T,) and EEG (T,) for a given load.
    * trials      - list of (eeg, pupil, meta) tuples
    * load        - 5 / 9 / 13
    * lag_samples - positive = EEG delayed (later) relative to pupil
    """
    fs = 100  # Hz

    ch_names, weights = load_weights(sub_id)
    if ch_names is None or weights is None:
        print(f"Missing or ambiguous weights for subject {sub_id:03d}, cannot plot.")
        return None
    if ch_names != FRONTAL_MIDLINE:
        print("Warning: channel names in weights do not match expected frontal midline set.")

    # Safe common window (handles WIN_OFFSET2 == 0)
    win = slice(WIN_OFFSET1, None if WIN_OFFSET2 == 0 else -WIN_OFFSET2)

    eeg_list, pupil_list = [], []

    for eeg, pupil, meta in trials:
        if meta.get("load") != load:
            continue

        # --- project EEG to 1D and window both signals ---
        eeg_roi = np.dot(eeg, weights).ravel()[win]      # (T,)
        p = np.asarray(pupil)
        pupil_1d = (p[:, 0] if p.ndim == 2 else p.ravel())[win]

        Tw = min(len(eeg_roi), len(pupil_1d))
        if Tw == 0:
            continue
        if abs(lag_samples) >= Tw:
            # Too big a lag for this window; skip this trial
            continue

        # --- asymmetric trimming (no roll, no wrap) ---
        if lag_samples >= 0:
            # EEG delayed: drop early EEG, drop late pupil
            eeg_out   = eeg_roi[lag_samples:]
            pupil_out = pupil_1d[:Tw - lag_samples]
        else:
            s = -lag_samples  # EEG advanced
            eeg_out   = eeg_roi[:Tw - s]
            pupil_out = pupil_1d[s:]

        # collect
        eeg_list.append(eeg_out)
        pupil_list.append(pupil_out)

    # No trials for this load
    if not eeg_list:
        return None

    # Crop each to the shortest length, stack, mean ± SEM
    L = min(map(len, eeg_list))
    eeg_mat   = np.stack([e[:L] for e in eeg_list], axis=0)
    pupil_mat = np.stack([p[:L] for p in pupil_list], axis=0)

    eeg_avg   = eeg_mat.mean(axis=0)
    eeg_sem   = eeg_mat.std(axis=0, ddof=1) / np.sqrt(eeg_mat.shape[0])
    pupil_avg = pupil_mat.mean(axis=0)
    pupil_sem = pupil_mat.std(axis=0, ddof=1) / np.sqrt(pupil_mat.shape[0])

    # Time axis: window starts at onset (because WIN_OFFSET1 == onset in your setup).
    # For lag < 0 (EEG advanced), our Option B trimming removes 's' samples from the START of pupil,
    # so the first common sample is s/fs after onset.
    t0 = 0.0 if lag_samples >= 0 else (-lag_samples) / fs
    t = t0 + np.arange(L) / fs

    return t, (eeg_avg, eeg_sem), (pupil_avg, pupil_sem)


import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ------------------------------------------------------------------ #
# ❶  Load the subject-level CCA table *once* at module import time
# ------------------------------------------------------------------ #
CCA_FILE = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\subject_level_cca_slow.csv")
CCA_DF   = pd.read_csv(CCA_FILE)        # columns: subject, condition, lag_ms, r, p-value
CCA_DF["condition"] = CCA_DF["condition"].str.lower()   # normalise to lowercase
CCA_DF.set_index(["subject", "condition"], inplace=True)

# ------------------------------------------------------------------ #
def plot_subject2(sub_id: int, loads=(5, 9, 13)):
    """
    One figure with 3 (rows = load) × 2 (cols = task) panels.
    Each panel overlays pupil and EEG-θ for a single load and task,
    and the panel title now includes the subject-specific CCA 'r'.
    """
    trials = load_all_trials(sub_id)
    mem, ctl = split_trials_by_condition(trials)

    lag       = best_shift_by_sub.get(sub_id, 0)  // 10          # samples @100 Hz
    colors    = {5: "#1f77b4", 9: "#ff7f0e", 13: "#2ca02c"}
    task_sets = [(mem, "memory"), (ctl, "control")]

    # ── look up the two r values for this subject ──────────────────
    r_memory  = float(CCA_DF.loc[(sub_id, "memory"), "r"])   if (sub_id, "memory")  in CCA_DF.index else float("nan")
    r_control = float(CCA_DF.loc[(sub_id, "control"), "r"])  if (sub_id, "control") in CCA_DF.index else float("nan")
    r_by_cond = {"memory": r_memory, "control": r_control}

    # ── make subplot grid (no shared x) ───────────────────────────
    fig, axes = plt.subplots(len(loads), 2, figsize=(12, 9), sharey="row")
    legend_handles, legend_labels = [], []

    for row, load in enumerate(loads):
        for col, (trial_set, cond) in enumerate(task_sets):
            ax = axes[row, col]

            res = trial_average2(sub_id, trial_set, load, cond, lag_samples=lag)
            if res is None:            # no data → hide the axis
                print(f"Subject {sub_id:02d} has no trials for load {load} ({cond})")
                ax.set_visible(False)
                continue

            t, (eeg_avg, eeg_sem), (pup_avg, pup_sem) = res
            # ─────────── z-score each series ──────────────────────────
            # (mean-remove, divide by SD; adjust SEM by the same SD)
            p_mean, p_std = pup_avg.mean(), pup_avg.std(ddof=0)
            e_mean, e_std = eeg_avg.mean(), eeg_avg.std(ddof=0)

            pup_avg = (pup_avg - p_mean) / p_std
            pup_sem = pup_sem / p_std

            eeg_avg = (eeg_avg - e_mean) / e_std
            eeg_sem = eeg_sem / e_std


            # pupil ────────────────────────────────────────────────
            ax.plot(t, pup_avg, color=colors[load], label="Pupil")
            ax.fill_between(t, pup_avg - pup_sem, pup_avg + pup_sem, color=colors[load], alpha=.25)

            # EEG θ ────────────────────────────────────────────────
            ax_r = ax.twinx()
            ax_r.plot(t, eeg_avg, color="black", lw=1.2, label="θ")
            ax_r.fill_between(t, eeg_avg - eeg_sem, eeg_avg + eeg_sem, color="black", alpha=.15)

            # cosmetics ───────────────────────────────────────────
            ax.set_xlim(0, t[-100])
            ax.set_yticks([]); ax_r.set_yticks([])
            ax.axvline(0, ls="--", c="k", lw=.7)

            if col == 0:
                ax.set_ylabel(f"Load {load:02d}")
            if row == 0:
                # add r to title
                r_val = r_by_cond[cond]
                ax.set_title(f"{cond.capitalize()}  (r = {r_val:.3f})")

            if not legend_handles:
                legend_handles += ax.get_lines() + ax_r.get_lines()
                legend_labels  += [h.get_label() for h in legend_handles]

    axes[0, 0].legend(legend_handles, legend_labels,
                      loc="upper right", frameon=False)

    fig.suptitle(f"Subject {sub_id:03d}   |   lag = {lag/100:.2f} s")
    fig.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()
    # ── SAVE the figure ───────────────────────────────────────────
    # ── SAVE instead of show ────────────────────────────────────────
    fname = PLOTS_DIR / f"s{sub_id:02d}.png"      #  e.g.  s05.png
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)    


In [ ]:
for subj in SUBJECTS:   # or any custom list
    plot_subject2(subj)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _project_and_align(eeg, pupil, weights, lag_samples=0):

    win = slice(WIN_OFFSET1, -WIN_OFFSET2)  # common window

    # --- project EEG, make pupil 1D, then window ---
    eeg_roi = np.dot(eeg, weights).ravel()[win]   # (T,)
    pupil   = np.asarray(pupil)
    pupil_1d_full = pupil[:, 0] if pupil.ndim == 2 else pupil.ravel()
    pupil_1d = pupil_1d_full[win]

    # --- common length & guards ---
    T = min(len(eeg_roi), len(pupil_1d))
    if T == 0:
        return np.empty(0), np.empty(0)
    if abs(lag_samples) >= T:
        return np.empty(0), np.empty(0)

    # --- asymmetric trimming (no roll, no wrap) ---
    if lag_samples >= 0:
        # EEG delayed: drop early EEG and late pupil
        eeg_roi  = eeg_roi[lag_samples:]
        pupil_1d = pupil_1d[:T - lag_samples]
    else:
        s = -lag_samples  # EEG advanced
        eeg_roi  = eeg_roi[:T - s]
        pupil_1d = pupil_1d[s:]

    return eeg_roi, pupil_1d

def _trialwise_z(x):
    mu = x.mean()
    sd = x.std(ddof=0)
    return (x - mu) / (sd if sd > 0 else 1.0)

def extract_trials_for_plot(
        subjects=None, loads=(5,9,13), z_per_trial=True, tasks = ["memory", "control"]
    ):
    """
    Pool *all trials* across all subjects for each (condition,load),
    then compute mean±SEM across trials (subjects with more trials weigh more).
    """
    # subjects default: infer from your CCA_DF or fallback to best_shift_by_sub keys
    if subjects is None:
        try:
            subjects = sorted({int(s) for (s, _) in CCA_DF.index})
        except Exception:
            subjects = sorted(best_shift_by_sub.keys()) if 'best_shift_by_sub' in globals() else []
    if not subjects:
        print("No subjects available.")
        return

    # For each panel we will collect lists of trial arrays
    panel_trials = { (cond, load): {"eeg": [], "pupil": []}
                     for cond in tasks for load in loads }

    # --- collect trials ---
    for sub in subjects:
        # per-subject pieces
        print(f"Loading subject {sub}...")
        try:
            trials = load_all_trials(sub)
        except Exception as e:
            print(f"Skipping subject {sub}: load_all_trials failed ({e})")
            continue

        try:
            ch_names, weights = load_weights(sub)
            if weights is None:
                print(f"Skipping subject {sub}: no weights.")
                continue
        except Exception as e:
            print(f"Skipping subject {sub}: load_weights failed ({e})")
            continue

        lag_ms = best_shift_by_sub.get(sub, 0) if 'best_shift_by_sub' in globals() else 0
        if lag_ms == 0: print(f"Subject {sub}: no lag")
        
        lag_samples = int(round(lag_ms / 10.0))   # 100 Hz → 10 ms per sample

        mem, ctl = split_trials_by_condition(trials)
        by_cond = {"memory": mem, "control": ctl}

        for cond in tasks:
            for eeg, pupil, meta in by_cond[cond]:
                L = meta.get("load", None)
                if L not in loads: 
                    continue

                eeg_roi, pupil_1d = _project_and_align(eeg, pupil, weights, lag_samples)

                if z_per_trial:
                    eeg_roi  = _trialwise_z(eeg_roi)
                    pupil_1d = _trialwise_z(pupil_1d)

                panel_trials[(cond, L)]["eeg"].append(eeg_roi)
                panel_trials[(cond, L)]["pupil"].append(pupil_1d)
    return panel_trials

def plot_grand_average_trials(panel_trials, tasks = ["memory", "control"], loads=(5,9,13), z_per_trial=True, save_name="grand_average_trials2.png"):
# --- plot 3×2 grid ---
    colors = {5: "#1f77b4", 9: "#ff7f0e", 13: "#2ca02c"}
    fig, axes = plt.subplots(len(loads), 2, figsize=(12, 9), sharey="row")
    legend_handles, legend_labels = [], []

    for r, load in enumerate(loads):
        for c, cond in enumerate(tasks):
            ax = axes[r, c]
            eeg_list  = panel_trials[(cond, load)]["eeg"]
            pup_list  = panel_trials[(cond, load)]["pupil"]

            if not eeg_list:
                ax.set_visible(False)
                continue

            # align all trials to the shortest length for this panel
            Lmin = min(map(len, eeg_list + pup_list))
            eeg_stack = np.stack([e[:Lmin] for e in eeg_list], axis=0)  # (Ntrials, L)
            pup_stack = np.stack([p[:Lmin] for p in pup_list], axis=0)

            # grand mean ± SEM across trials
            eeg_mu  = eeg_stack.mean(0)
            eeg_sem = eeg_stack.std(0, ddof=1) / np.sqrt(eeg_stack.shape[0])

            pup_mu  = pup_stack.mean(0)
            pup_sem = pup_stack.std(0, ddof=1) / np.sqrt(pup_stack.shape[0])

            # time axis (assumes 100 Hz)
            t = np.arange(Lmin) / 100.0

            ln1, = ax.plot(t, pup_mu, color=colors[load], label="Pupil")
            ax.fill_between(t, pup_mu - pup_sem, pup_mu + pup_sem, color=colors[load], alpha=.25)

            axr = ax.twinx()
            ln2, = axr.plot(t, eeg_mu, color="black", lw=1.2, label="θ")
            axr.fill_between(t, eeg_mu - eeg_sem, eeg_mu + eeg_sem, color="black", alpha=.15)

            # ✅ use the actual time range of your data
            ax.set_xlim(t[0], t[-1])
            ax.set_yticks([]); axr.set_yticks([])
            ax.axvline(0, ls="--", c="k", lw=.7)

            if c == 0:
                ax.set_ylabel(f"Load {load:02d}")
            if r == 0:
                n_tr = eeg_stack.shape[0]
                ax.set_title(f"{cond.capitalize()}")

            if not legend_handles:
                legend_handles.extend([ln1, ln2])
                legend_labels.extend(["Pupil", "θ"])

    axes[0, 0].legend(legend_handles, legend_labels, loc="upper right", frameon=False)
    total_trials = sum(len(panel_trials[k]["eeg"]) for k in panel_trials)
    #fig.suptitle(f"Grand average across trials (total N={total_trials}, z_per_trial={z_per_trial})")
    fig.tight_layout(rect=[0, 0.03, 1, 0.97])

    #out_path = PLOTS_DIR / save_name
    #fig.savefig(out_path, dpi=300, bbox_inches="tight")
    #plt.close(fig)
    #print(f"Saved: {out_path}")


In [ ]:
panel_trials = extract_trials_for_plot()

In [ ]:
plot_grand_average_trials(panel_trials)

In [ ]:
import pandas as pd
import glob
from pathlib import Path

# >>> change nothing below unless you need to tweak <<<
ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\cca_weights_fast")  # adjust to your weights directory

# Match: sub-xxx_eeg_weights_mem_....csv  (xxx and the trailing part can be anything)
pattern = str(ROOT / r"sub-*_eeg_weights_mem_*.csv")  # note: double underscore only if your files have it
# If your files actually use a single underscore, use this instead:
# pattern = str(ROOT / r"sub-*_eeg_weights_mem_*.csv")

files = glob.glob(pattern)

if not files:
    raise FileNotFoundError(f"No files matched pattern:\n{pattern}\n"
                            "If your files have a different underscore pattern, switch the pattern line.")

dfs = []
for fp in files:
    df = pd.read_csv(fp)
    # normalize column names just in case
    df.columns = [c.strip().lower() for c in df.columns]
    if not {"channel", "weight"} <= set(df.columns):
        raise ValueError(f"{fp} is missing required columns 'channel' and 'weight'")
    # keep only what's needed
    dfs.append(df[["channel", "weight"]].copy())

all_weights = pd.concat(dfs, ignore_index=True)

summary = (
    all_weights
    .groupby("channel", as_index=True)["weight"]
    .agg(mean="mean", std="std", count="count")
    .sort_index()
)

out_path = ROOT / "avg_eeg_weights_mem.csv"
summary.to_csv(out_path)

print(f"Read {len(files)} files.")
print(f"Saved per-electrode summary to: {out_path}")
print(summary.head(10))


In [ ]:
%matplotlib qt
# %matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
# Use Arial everywhere (falls back if not available)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,          # base font size
    "axes.titlesize": 16,     # figure/axes titles
    "axes.labelsize": 13,     # x/y labels
    "legend.fontsize": 11,    # legend text
    "xtick.labelsize": 11,    # tick labels
    "ytick.labelsize": 11,
    "figure.titlesize": 18,   # suptitle
})

In [ ]:
import csv
from pathlib import Path

import numpy as np


# Try to use SciPy's griddata for smooth interpolation; fallback to IDW if unavailable.
try:
    from scipy.interpolate import griddata
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# --- Configuration ---
CSV_PATH = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\cca_weights_slow\avg_eeg_weights_mem.csv")
OUT_PATH = CSV_PATH.with_name("avg_eeg_weights_mem_topomap.png")

# Approximate 10–20 2D coordinates on a unit head (front = +y, left = -x)
# These are schematic positions good enough for qualitative topomaps.
POS = {
    "AFz": (0.0,  0.75),
    "AF3": (-0.35, 0.70),
    "AF4": ( 0.35, 0.70),

    "Fz":  (0.0,  0.55),
    "F1":  (-0.15, 0.52),
    "F2":  ( 0.15, 0.52),
    "F3":  (-0.30, 0.50),
    "F4":  ( 0.30, 0.50),

    "FC1": (-0.18, 0.32),
    "FC2": ( 0.18, 0.32),
    "FC3": (-0.35, 0.30),
    "FC4": ( 0.35, 0.30),

    "Cz":  (0.0,  0.10),
    "C1":  (-0.18, 0.10),
    "C2":  ( 0.18, 0.10),
    "C3":  (-0.36, 0.08),
    "C4":  ( 0.36, 0.08),

    # (Add more labels here if present in your CSVs)
}

# --- Load CSV (expects columns: channel, weight) ---
names, vals = [], []
with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    weight_col = "mean"

    # read rows
    for row in reader:
        ch = (row.get("channel")).strip()
        if not ch:
            continue
        try:
            w = float(row.get(weight_col))
        except Exception:
            # row might have original casing; try more robustly
            w = float(row[[k for k in row if k.lower() == weight_col][0]])
        if ch in POS:
            names.append(ch)
            vals.append(w)

if not names:
    raise RuntimeError("No channels from the CSV matched the known scalp positions in POS. "
                       "Add the missing labels to POS.")

xy = np.array([POS[ch] for ch in names])
z = np.array(vals, dtype=float)

mu = z.mean()
sd = z.std(ddof=0)
#if sd == 0:
#    z = np.zeros_like(z)  # all equal -> all zeros after standardization
#else:
#    z = (z - mu) / sd

# --- Build grid on the unit head circle ---
N = 200
grid_x = np.linspace(-1.0, 1.0, N)
grid_y = np.linspace(-1.0, 1.0, N)
GX, GY = np.meshgrid(grid_x, grid_y)
mask = (GX**2 + GY**2) <= 1.0  # inside head

# --- Interpolate to grid ---
def idw_interpolate(points, values, gx, gy, power=2.0, eps=1e-12):
    """Inverse-distance weighting as a fallback (no SciPy)."""
    px = points[:, 0][:, None, None]
    py = points[:, 1][:, None, None]
    dx = gx[None, :, :] - px
    dy = gy[None, :, :] - py
    d2 = dx*dx + dy*dy
    w = 1.0 / (d2 + eps)**(power/2)
    num = (w * values[:, None, None]).sum(axis=0)
    den = w.sum(axis=0)
    return num / (den + eps)

if _HAS_SCIPY:
    GZ = griddata(xy, z, (GX, GY), method="cubic")
    # fill NaNs (outside convex hull) via IDW to avoid holes at edges
    nan_mask = np.isnan(GZ)
    if nan_mask.any():
        GZ_idw = idw_interpolate(xy, z, GX, GY, power=2.0)
        GZ[nan_mask] = GZ_idw[nan_mask]
else:
    GZ = idw_interpolate(xy, z, GX, GY, power=2.0)

# mask outside head
GZ_masked = np.ma.array(GZ, mask=~mask)

# --- Plot ---
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(
    GZ_masked,
    origin="lower",
    extent=(-1, 1, -1, 1),
    interpolation="bilinear",
)
# head outline
theta = np.linspace(0, 2*np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), linewidth=2)

# simple nose
nose_x = np.array([ -0.06, 0.0, 0.06 ])
nose_y = np.array([  1.00, 1.08, 1.00 ])
ax.plot(nose_x, nose_y, linewidth=2)

# ears
ear_x = np.array([1.02, 1.07, 1.02])
ear_y = np.array([ 0.10, 0.00, -0.10])
ax.plot( ear_x,  ear_y, linewidth=2)
ax.plot(-ear_x,  ear_y, linewidth=2)

# electrodes as dots
ex, ey = xy[:, 0], xy[:, 1]
ax.scatter(ex, ey, s=25, edgecolors="k")
for ch, (cx, cy) in POS.items():
    if ch in names:
        ax.text(cx, cy, ch, ha="center", va="bottom", fontsize=8)

cbar = plt.colorbar(im, ax=ax)
#cbar.set_label("Average weight")

ax.set_aspect("equal")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.15, 1.15)
ax.set_xticks([])
ax.set_yticks([])
#ax.set_title("Elastic CCA — Average EEG weights (memory)")

fig.tight_layout()
fig.savefig(OUT_PATH, dpi=200)
print(f"Saved scalp map to: {OUT_PATH}")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Load ---
path = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\subject_level_cca_slow.csv"
# --- Load ---
path_fast = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\new_roll\subject_level_cca_fast.csv"

mode = "fast"  # "slow" or "fast"
if mode == "fast":
    df = pd.read_csv(path_fast)
    limit = 1000
    bin = 100
else:
    df = pd.read_csv(path)
    limit = 2000
    bin = 200

# --------- HISTOGRAM ----------
df = df[df["condition"].eq("memory")].copy()
lags = df["lag_ms"].to_numpy(dtype=float)

# bins every 200 ms from -1000 to 1000 (adjust if needed)
bins = np.arange(-limit, limit + bin, bin)

fig, ax = plt.subplots(figsize=(3.5, 2))
ax.hist(lags, bins=bins, edgecolor="k", alpha=0.85)
ax.axvline(0, color="k", lw=1, ls="--", alpha=0.7)

# summary stats
mu = np.mean(lags)
med = np.median(lags)
n = len(lags)
ax.axvline(mu, color="C1", lw=1.2, ls="--")
ax.axvline(med, color="C2", lw=1.2, ls=":")
ax.set_xlim(bins[0], bins[-1])

ax.set_xlabel("Optimal lag (ms)")
ax.set_ylabel("Subjects")
#ax.set_title(f"Subject-wise CCA optimal lag —  (n={n})")
#ax.text(0.01, 0.98, f"mean={mu:.0f} ms\nmedian={med:.0f} ms", transform=ax.transAxes, ha="left", va="top")

plt.tight_layout()
out = Path(f"lag_hist_{mode}.png")
plt.savefig(out, dpi=200, bbox_inches="tight")
print(f"Saved: {out}")
print(f"Mean lag: {mu:.1f} ms, Median lag: {med:.1f} ms")